# Predictive ATM IV Gap and Deterioration Diagnostics

This notebook-level script checks whether low observed ATM implied volatility
periods coincide with a larger one-step-ahead model-minus-observed ATM IV gap.
It then checks whether that gap helps account for predictive deterioration.

The diagnostic uses the main q160-update/q300-evaluation thesis runs. The model
ATM IV is obtained from the pre-update predictive mean option price, inverted
using the observed forward convention on the same selected quotes. This makes
the comparison a quote-space predictive diagnostic rather than a filtered-state
level diagnostic.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import ndtr

warnings.filterwarnings("ignore", category=RuntimeWarning)

def find_project_root(start=None):
    here = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/ and notebooks/.")


def project_relative(path):
    path = Path(path)
    try:
        return path.relative_to(PROJECT_ROOT)
    except ValueError:
        return path


PROJECT_ROOT = find_project_root()
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
ANALYSIS_DIR = OUTPUTS_DIR / "comparisons" / "predictive_atm_iv_gap_deterioration"
TABLE_DIR = ANALYSIS_DIR / "tables"
FIG_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [123, 456, 789, 101112, 131415]
TARGET_MATURITIES_DAYS = [7.0, 30.0]
N_ATM_QUOTES = 6
MAX_NEAREST_ABS_MONEYNESS = 0.15
HAC_LAG = 5

WINDOWS = [
    {
        "label": "0--399",
        "normal": OUTPUTS_DIR / "common_eval_q160_eval300" / "normal_400ts_q160_eval300_seed{seed}",
        "rough": OUTPUTS_DIR / "common_eval_q160_eval300" / "rough_400ts_q160_eval300_seed{seed}",
    },
    {
        "label": "400--799",
        "normal": OUTPUTS_DIR / "nonoverlap_q160_eval300" / "w401_800" / "normal_sabr_no_A_RW_seed{seed}",
        "rough": OUTPUTS_DIR / "nonoverlap_q160_eval300" / "w401_800" / "rough_sabr_logU0_seed{seed}",
    },
    {
        "label": "800--1199",
        "normal": OUTPUTS_DIR / "nonoverlap_q160_eval300" / "w801_1200" / "normal_sabr_no_A_RW_seed{seed}",
        "rough": OUTPUTS_DIR / "nonoverlap_q160_eval300" / "w801_1200" / "rough_sabr_logU0_seed{seed}",
    },
    {
        "label": "1200--1599",
        "normal": OUTPUTS_DIR / "nonoverlap_q160_eval300" / "w1201_1600" / "normal_sabr_no_A_RW_seed{seed}",
        "rough": OUTPUTS_DIR / "nonoverlap_q160_eval300" / "w1201_1600" / "rough_sabr_logU0_seed{seed}",
    },
]

MODEL_LABELS = {
    "normal": "Normal SABR",
    "rough": "Rough-SABR",
}

## Helpers

The fixed-maturity ATM extraction follows the earlier filtered-mean IV notebook:
within each expiry and timestamp, use the six quotes closest to ATM, then
interpolate the resulting expiry-level ATM quantity to fixed 7-day and 30-day
maturities.

In [ ]:
def normalise_iv_units(series):
    values = pd.to_numeric(series, errors="coerce").astype(float)
    finite = values[np.isfinite(values)]
    if len(finite) and finite.median() > 5.0:
        values = values / 100.0
    return values


def black76_price_usd(forward, strike, maturity, sigma, option_type):
    forward = np.asarray(forward, dtype=float)
    strike = np.asarray(strike, dtype=float)
    maturity = np.asarray(maturity, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    option_type = np.asarray(option_type).astype(str)

    price = np.full(np.broadcast(forward, strike, maturity, sigma).shape, np.nan, dtype=float)
    valid = (forward > 0.0) & (strike > 0.0) & (maturity > 0.0) & (sigma > 0.0)
    if not np.any(valid):
        return price

    vol_sqrt_t = sigma[valid] * np.sqrt(maturity[valid])
    d1 = (np.log(forward[valid] / strike[valid]) + 0.5 * sigma[valid] ** 2 * maturity[valid]) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    is_call = np.char.startswith(np.char.lower(option_type[valid].astype(str)), "c")
    price[valid] = np.where(
        is_call,
        forward[valid] * ndtr(d1) - strike[valid] * ndtr(d2),
        strike[valid] * ndtr(-d2) - forward[valid] * ndtr(-d1),
    )
    return price


def implied_vol_from_btc_price(price_btc, forward, strike, maturity, option_type, max_vol=5.0, tol=1e-7, n_iter=80):
    """Invert BTC-premium Black-76 prices after converting to USD with the supplied forward."""
    price_btc = np.asarray(price_btc, dtype=float)
    forward = np.asarray(forward, dtype=float)
    strike = np.asarray(strike, dtype=float)
    maturity = np.asarray(maturity, dtype=float)
    option_type = np.asarray(option_type).astype(str)

    target_usd = price_btc * forward
    is_call = np.char.startswith(np.char.lower(option_type.astype(str)), "c")
    intrinsic = np.where(is_call, np.maximum(forward - strike, 0.0), np.maximum(strike - forward, 0.0))
    high_price = black76_price_usd(forward, strike, maturity, np.full_like(forward, max_vol), option_type)

    valid = (
        np.isfinite(target_usd)
        & np.isfinite(high_price)
        & (forward > 0.0)
        & (strike > 0.0)
        & (maturity > 0.0)
        & (target_usd >= intrinsic - tol)
        & (target_usd <= high_price + tol)
    )

    out = np.full_like(target_usd, np.nan, dtype=float)
    if not np.any(valid):
        return out

    lo = np.full(np.sum(valid), 1e-6, dtype=float)
    hi = np.full(np.sum(valid), max_vol, dtype=float)
    f = forward[valid]
    k = strike[valid]
    t = maturity[valid]
    cp = option_type[valid]
    target = np.maximum(target_usd[valid], intrinsic[valid])

    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        mid_price = black76_price_usd(f, k, t, mid, cp)
        too_low = mid_price < target
        lo = np.where(too_low, mid, lo)
        hi = np.where(too_low, hi, mid)

    out[valid] = 0.5 * (lo + hi)
    return out


def read_csv_columns(path, wanted):
    header = pd.read_csv(path, nrows=0)
    usecols = [col for col in wanted if col in header.columns]
    missing = sorted(set(wanted) - set(usecols))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    return pd.read_csv(path, usecols=usecols)


def nearest_atm_rows(panel):
    panel = panel.copy()
    panel["abs_log_moneyness"] = panel["log_moneyness"].abs()
    keep_idx = []
    for _, group in panel.groupby(["t_index", "expiry"], sort=False):
        if not len(group):
            continue
        nearest = group.nsmallest(min(N_ATM_QUOTES, len(group)), "abs_log_moneyness")
        if nearest["abs_log_moneyness"].min() <= MAX_NEAREST_ABS_MONEYNESS:
            keep_idx.extend(nearest.index.to_list())
    return panel.loc[keep_idx].copy()


def expiry_atm_points(near):
    rows = []
    group_cols = ["window", "model", "seed", "t_index", "capture_time_utc", "expiry"]
    for key, group in near.groupby(group_cols, sort=True):
        group = group[np.isfinite(group["market_iv_decimal"]) & np.isfinite(group["model_iv_decimal"])]
        if group.empty:
            continue
        base = dict(zip(group_cols, key))
        rows.append({
            **base,
            "T_years": float(group["T_years"].median()),
            "observed_atm_iv_percent": float(100.0 * group["market_iv_decimal"].median()),
            "model_atm_iv_percent": float(100.0 * group["model_iv_decimal"].median()),
            "observed_atm_price_btc": float(group["market_price"].median()),
            "model_atm_price_btc": float(group["predictive_price_mean"].median()),
            "n_quotes_used": int(len(group)),
            "nearest_abs_log_moneyness": float(group["abs_log_moneyness"].min()),
        })
    return pd.DataFrame(rows)


def interpolate_fixed_maturity(points, target_days):
    if len(points) < 2:
        return None
    target_years = target_days / 365.0
    points = points.sort_values("T_years")
    maturities = points["T_years"].to_numpy(dtype=float)
    if target_years < np.nanmin(maturities) or target_years > np.nanmax(maturities):
        return None

    metrics = [
        "observed_atm_iv_percent",
        "model_atm_iv_percent",
        "observed_atm_price_btc",
        "model_atm_price_btc",
    ]
    row = {
        "target_days": target_days,
        "n_expiries_for_interp": int(len(points)),
        "nearest_abs_log_moneyness": float(points["nearest_abs_log_moneyness"].min()),
    }
    for col in metrics:
        values = points[col].to_numpy(dtype=float)
        valid = np.isfinite(maturities) & np.isfinite(values)
        if valid.sum() < 2:
            row[col] = np.nan
        else:
            row[col] = float(np.interp(target_years, maturities[valid], values[valid]))
    return row


def fixed_maturity_series(expiry_points):
    rows = []
    id_cols = ["window", "model", "seed", "t_index", "capture_time_utc"]
    for key, panel in expiry_points.groupby(id_cols, sort=True):
        base = dict(zip(id_cols, key))
        for target_days in TARGET_MATURITIES_DAYS:
            row = interpolate_fixed_maturity(panel, target_days)
            if row is not None:
                rows.append({**base, **row})
    fixed = pd.DataFrame(rows)
    if fixed.empty:
        return fixed
    fixed["atm_iv_gap_pp"] = fixed["model_atm_iv_percent"] - fixed["observed_atm_iv_percent"]
    fixed["atm_price_error_btc"] = fixed["model_atm_price_btc"] - fixed["observed_atm_price_btc"]
    return fixed

## Load the predictive runs

In [ ]:
def load_predictive_run(folder, model_label, seed, window_label):
    folder = Path(folder)
    pred_path = folder / "predictive_option_prices_eval.csv"
    ll_path = folder / "predictive_loglikelihood_eval.csv"
    if not pred_path.exists():
        raise FileNotFoundError(pred_path)
    if not ll_path.exists():
        raise FileNotFoundError(ll_path)

    pred_cols = [
        "capture_time_utc",
        "t_index",
        "expiry",
        "T_years",
        "strike",
        "option_type_clean",
        "F_obs",
        "market_price",
        "market_iv",
        "mark_iv",
        "log_moneyness",
        "predictive_price_mean",
    ]
    pred = read_csv_columns(pred_path, pred_cols)
    for col in ["t_index", "T_years", "strike", "F_obs", "market_price", "market_iv", "mark_iv", "log_moneyness", "predictive_price_mean"]:
        pred[col] = pd.to_numeric(pred[col], errors="coerce")
    pred["window"] = window_label
    pred["model"] = model_label
    pred["seed"] = seed
    pred["market_iv_decimal"] = normalise_iv_units(pred["market_iv"])

    pred = pred[
        np.isfinite(pred["t_index"])
        & np.isfinite(pred["T_years"])
        & np.isfinite(pred["F_obs"])
        & np.isfinite(pred["strike"])
        & np.isfinite(pred["market_price"])
        & np.isfinite(pred["predictive_price_mean"])
        & np.isfinite(pred["log_moneyness"])
    ].copy()

    near = nearest_atm_rows(pred)
    near["model_iv_decimal"] = implied_vol_from_btc_price(
        near["predictive_price_mean"].to_numpy(),
        near["F_obs"].to_numpy(),
        near["strike"].to_numpy(),
        near["T_years"].to_numpy(),
        near["option_type_clean"].to_numpy(),
    )

    expiry_points = expiry_atm_points(near)
    fixed = fixed_maturity_series(expiry_points)

    ll = read_csv_columns(ll_path, ["t_index", "avg_log_predictive_likelihood_per_quote"])
    ll["t_index"] = pd.to_numeric(ll["t_index"], errors="coerce")
    ll["deterioration_per_quote"] = -pd.to_numeric(
        ll["avg_log_predictive_likelihood_per_quote"],
        errors="coerce",
    )
    fixed = fixed.merge(
        ll[["t_index", "deterioration_per_quote"]],
        on="t_index",
        how="left",
    )
    fixed["valid_model_iv_share_near_atm"] = float(np.isfinite(near["model_iv_decimal"]).mean())
    fixed["source_folder"] = str(folder.relative_to(PROJECT_ROOT))
    return fixed


all_fixed_seed = []
for window in WINDOWS:
    for seed in SEEDS:
        for model_key, model_label in MODEL_LABELS.items():
            folder = Path(str(window[model_key]).format(seed=seed))
            fixed = load_predictive_run(folder, model_label, seed, window["label"])
            all_fixed_seed.append(fixed)

fixed_seed = pd.concat(all_fixed_seed, ignore_index=True)
fixed_seed = fixed_seed.sort_values(["target_days", "window", "model", "seed", "t_index"])
print(f"Built {len(fixed_seed):,} fixed-maturity seed-level rows.")
fixed_seed.head()

## Aggregate over paired seeds

The regressions use timestamp-level means across the five paired seeds. This
matches the spirit of the thesis HAC score analysis and avoids treating the
same market timestamp as five independent observations.

In [ ]:
timestamp = (
    fixed_seed
    .groupby(["window", "model", "target_days", "t_index"], as_index=False)
    .agg(
        capture_time_utc=("capture_time_utc", "first"),
        observed_atm_iv_percent=("observed_atm_iv_percent", "mean"),
        model_atm_iv_percent=("model_atm_iv_percent", "mean"),
        model_atm_iv_std_percent=("model_atm_iv_percent", "std"),
        atm_iv_gap_pp=("atm_iv_gap_pp", "mean"),
        observed_atm_price_btc=("observed_atm_price_btc", "mean"),
        model_atm_price_btc=("model_atm_price_btc", "mean"),
        atm_price_error_btc=("atm_price_error_btc", "mean"),
        deterioration_per_quote=("deterioration_per_quote", "mean"),
        seeds_available=("atm_iv_gap_pp", lambda x: int(np.isfinite(x).sum())),
        n_expiries_for_interp=("n_expiries_for_interp", "median"),
        nearest_abs_log_moneyness=("nearest_abs_log_moneyness", "median"),
        valid_model_iv_share_near_atm=("valid_model_iv_share_near_atm", "mean"),
    )
)

wide = timestamp.pivot_table(
    index=["window", "target_days", "t_index", "capture_time_utc"],
    columns="model",
    values=["observed_atm_iv_percent", "atm_iv_gap_pp", "atm_price_error_btc", "deterioration_per_quote"],
    aggfunc="first",
).reset_index()
wide.columns = [
    "_".join([str(part) for part in col if str(part)])
    if isinstance(col, tuple)
    else str(col)
    for col in wide.columns
]
wide["observed_atm_iv_percent"] = wide[[
    "observed_atm_iv_percent_Normal SABR",
    "observed_atm_iv_percent_Rough-SABR",
]].mean(axis=1)
wide["mean_atm_iv_gap_pp"] = wide[[
    "atm_iv_gap_pp_Normal SABR",
    "atm_iv_gap_pp_Rough-SABR",
]].mean(axis=1)
wide["common_deterioration_per_quote"] = wide[[
    "deterioration_per_quote_Normal SABR",
    "deterioration_per_quote_Rough-SABR",
]].mean(axis=1)

audit = (
    timestamp
    .groupby(["target_days", "model"], as_index=False)
    .agg(
        n_timestamps=("t_index", "nunique"),
        mean_observed_atm_iv_percent=("observed_atm_iv_percent", "mean"),
        mean_model_atm_iv_percent=("model_atm_iv_percent", "mean"),
        mean_gap_pp=("atm_iv_gap_pp", "mean"),
        mean_abs_gap_pp=("atm_iv_gap_pp", lambda x: float(np.mean(np.abs(x)))),
        mean_price_error_btc=("atm_price_error_btc", "mean"),
        mean_abs_price_error_btc=("atm_price_error_btc", lambda x: float(np.mean(np.abs(x)))),
        mean_deterioration_per_quote=("deterioration_per_quote", "mean"),
        mean_seeds_available=("seeds_available", "mean"),
        mean_valid_model_iv_share_near_atm=("valid_model_iv_share_near_atm", "mean"),
    )
)

fixed_seed_path = TABLE_DIR / "predictive_fixed_maturity_atm_iv_gap_seed_level.csv"
timestamp_path = TABLE_DIR / "predictive_fixed_maturity_atm_iv_gap_timestamp_mean.csv"
wide_path = TABLE_DIR / "predictive_fixed_maturity_atm_iv_gap_wide_timestamp_mean.csv"
audit_path = TABLE_DIR / "predictive_fixed_maturity_atm_iv_gap_summary.csv"

fixed_seed.to_csv(fixed_seed_path, index=False)
timestamp.to_csv(timestamp_path, index=False)
wide.to_csv(wide_path, index=False)
audit.to_csv(audit_path, index=False)

print(f"Saved {fixed_seed_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {timestamp_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {wide_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {audit_path.relative_to(PROJECT_ROOT)}")
audit

## HAC regression helpers

All regression variables are standardised before estimation; window fixed
effects are included but not reported in the compact tables.

In [ ]:
def standardize_in_place(df, cols):
    for col in cols:
        values = pd.to_numeric(df[col], errors="coerce").astype(float)
        mu = values.mean()
        sd = values.std(ddof=0)
        if not np.isfinite(sd) or sd == 0.0:
            df[col] = 0.0
        else:
            df[col] = (values - mu) / sd
    return df


def ols_hac(data, y_col, x_cols, fe_col="window", hac_lag=5):
    cols = [y_col] + x_cols + [fe_col, "t_index"]
    d = data[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    d = d.sort_values("t_index").reset_index(drop=True)
    n = len(d)
    if n <= len(x_cols) + 2:
        return pd.DataFrame()

    d = standardize_in_place(d, [y_col] + x_cols)
    fe = pd.get_dummies(d[fe_col].astype(str), prefix=fe_col, drop_first=True, dtype=float)
    X_df = pd.concat(
        [
            pd.Series(1.0, index=d.index, name="const"),
            d[x_cols].astype(float),
            fe,
        ],
        axis=1,
    )
    X = X_df.to_numpy(dtype=float)
    y = d[y_col].to_numpy(dtype=float)
    xtx_inv = np.linalg.pinv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    resid = y - X @ beta

    xe = X * resid[:, None]
    s_mat = xe.T @ xe
    max_lag = min(hac_lag, n - 1)
    for lag in range(1, max_lag + 1):
        weight = 1.0 - lag / (max_lag + 1.0)
        gamma = xe[lag:].T @ xe[:-lag]
        s_mat += weight * (gamma + gamma.T)

    cov = xtx_inv @ s_mat @ xtx_inv
    se = np.sqrt(np.maximum(np.diag(cov), 0.0))
    t_stat = np.divide(beta, se, out=np.full_like(beta, np.nan), where=se > 0.0)
    p_value = 2.0 * (1.0 - ndtr(np.abs(t_stat)))
    fitted = X @ beta
    ssr = float(np.sum((y - fitted) ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1.0 - ssr / sst if sst > 0.0 else np.nan

    rows = []
    for name, b, s, t, p in zip(X_df.columns, beta, se, t_stat, p_value):
        rows.append({
            "term": name,
            "coef": float(b),
            "hac_se": float(s),
            "t_stat": float(t),
            "p_value": float(p),
            "n_obs": int(n),
            "r2": float(r2),
            "hac_lag": int(max_lag),
        })
    return pd.DataFrame(rows)


def collect_regression(data, y_col, x_cols, spec, model=None, target_days=None):
    result = ols_hac(data, y_col, x_cols, hac_lag=HAC_LAG)
    if result.empty:
        return result
    result.insert(0, "spec", spec)
    result.insert(0, "target_days", target_days)
    result.insert(0, "model", model)
    result.insert(0, "y", y_col)
    return result[result["term"].isin(x_cols)].copy()


def format_p(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "<0.001"
    return f"{p:.3f}"

## Regressions

The first table asks whether the model-minus-observed ATM IV gap is larger
when observed ATM IV is lower. The second table links the IV gap to the ATM
price error. The third table checks whether the observed ATM IV coefficient
shrinks once the IV gap is included in a deterioration regression.

In [ ]:
gap_regs = []
price_regs = []
deterioration_regs = []

for target_days in TARGET_MATURITIES_DAYS:
    for model in ["Normal SABR", "Rough-SABR"]:
        panel = timestamp[
            (timestamp["target_days"] == target_days)
            & (timestamp["model"] == model)
        ].copy()

        gap_regs.append(collect_regression(
            panel,
            "atm_iv_gap_pp",
            ["observed_atm_iv_percent"],
            "gap_on_observed_iv",
            model,
            target_days,
        ))
        price_regs.append(collect_regression(
            panel,
            "atm_price_error_btc",
            ["atm_iv_gap_pp"],
            "price_error_on_gap",
            model,
            target_days,
        ))
        price_regs.append(collect_regression(
            panel,
            "atm_price_error_btc",
            ["observed_atm_iv_percent", "atm_iv_gap_pp"],
            "price_error_on_observed_iv_and_gap",
            model,
            target_days,
        ))
        deterioration_regs.append(collect_regression(
            panel,
            "deterioration_per_quote",
            ["observed_atm_iv_percent"],
            "deterioration_on_observed_iv",
            model,
            target_days,
        ))
        deterioration_regs.append(collect_regression(
            panel,
            "deterioration_per_quote",
            ["atm_iv_gap_pp"],
            "deterioration_on_gap",
            model,
            target_days,
        ))
        deterioration_regs.append(collect_regression(
            panel,
            "deterioration_per_quote",
            ["observed_atm_iv_percent", "atm_iv_gap_pp"],
            "deterioration_on_observed_iv_and_gap",
            model,
            target_days,
        ))

gap_regs = pd.concat(gap_regs, ignore_index=True)
price_regs = pd.concat(price_regs, ignore_index=True)
deterioration_regs = pd.concat(deterioration_regs, ignore_index=True)

common_regs = []
for target_days in TARGET_MATURITIES_DAYS:
    panel = wide[wide["target_days"] == target_days].copy()
    common_regs.append(collect_regression(
        panel,
        "common_deterioration_per_quote",
        ["observed_atm_iv_percent"],
        "common_deterioration_on_observed_iv",
        "Common",
        target_days,
    ))
    common_regs.append(collect_regression(
        panel,
        "common_deterioration_per_quote",
        ["observed_atm_iv_percent", "mean_atm_iv_gap_pp"],
        "common_deterioration_on_observed_iv_and_mean_gap",
        "Common",
        target_days,
    ))
common_regs = pd.concat(common_regs, ignore_index=True)

gap_reg_path = TABLE_DIR / "gap_on_observed_atm_iv_hac_regressions.csv"
price_reg_path = TABLE_DIR / "atm_price_error_gap_hac_regressions.csv"
deterioration_reg_path = TABLE_DIR / "deterioration_iv_gap_hac_regressions.csv"
common_reg_path = TABLE_DIR / "common_deterioration_mean_gap_hac_regressions.csv"

gap_regs.to_csv(gap_reg_path, index=False)
price_regs.to_csv(price_reg_path, index=False)
deterioration_regs.to_csv(deterioration_reg_path, index=False)
common_regs.to_csv(common_reg_path, index=False)

print(f"Saved {gap_reg_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {price_reg_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {deterioration_reg_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {common_reg_path.relative_to(PROJECT_ROOT)}")

gap_regs

## Coefficient shrinkage summary

This table is the compact mechanism check. The shrinkage column reports how
much the absolute observed-ATM-IV coefficient falls after adding the ATM IV gap.

In [ ]:
rows = []
for target_days in TARGET_MATURITIES_DAYS:
    for model in ["Normal SABR", "Rough-SABR"]:
        base = deterioration_regs[
            (deterioration_regs["target_days"] == target_days)
            & (deterioration_regs["model"] == model)
            & (deterioration_regs["spec"] == "deterioration_on_observed_iv")
            & (deterioration_regs["term"] == "observed_atm_iv_percent")
        ].iloc[0]
        full_obs = deterioration_regs[
            (deterioration_regs["target_days"] == target_days)
            & (deterioration_regs["model"] == model)
            & (deterioration_regs["spec"] == "deterioration_on_observed_iv_and_gap")
            & (deterioration_regs["term"] == "observed_atm_iv_percent")
        ].iloc[0]
        full_gap = deterioration_regs[
            (deterioration_regs["target_days"] == target_days)
            & (deterioration_regs["model"] == model)
            & (deterioration_regs["spec"] == "deterioration_on_observed_iv_and_gap")
            & (deterioration_regs["term"] == "atm_iv_gap_pp")
        ].iloc[0]
        shrink = 1.0 - abs(full_obs["coef"]) / abs(base["coef"]) if base["coef"] != 0 else np.nan
        rows.append({
            "target_days": target_days,
            "model": model,
            "observed_iv_coef_alone": base["coef"],
            "observed_iv_p_alone": base["p_value"],
            "observed_iv_coef_with_gap": full_obs["coef"],
            "observed_iv_p_with_gap": full_obs["p_value"],
            "gap_coef_with_observed_iv": full_gap["coef"],
            "gap_p_with_observed_iv": full_gap["p_value"],
            "absolute_observed_iv_coef_shrinkage": shrink,
            "n_obs": int(base["n_obs"]),
        })

shrinkage = pd.DataFrame(rows)
shrinkage_path = TABLE_DIR / "deterioration_observed_iv_coefficient_shrinkage.csv"
shrinkage.to_csv(shrinkage_path, index=False)
print(f"Saved {shrinkage_path.relative_to(PROJECT_ROOT)}")
shrinkage

## Binned visual diagnostic

The plot bins timestamps by observed fixed-maturity ATM IV and shows the
average model-minus-observed ATM IV gap in each bin. It is intentionally
less dense than a full timestamp plot.

In [ ]:
colors = {"Normal SABR": "#2563eb", "Rough-SABR": "#dc2626"}

fig, axes = plt.subplots(1, len(TARGET_MATURITIES_DAYS), figsize=(12.2, 4.5), sharey=True)
if len(TARGET_MATURITIES_DAYS) == 1:
    axes = [axes]

binned_rows = []
for ax, target_days in zip(axes, TARGET_MATURITIES_DAYS):
    for model in ["Normal SABR", "Rough-SABR"]:
        panel = timestamp[
            (timestamp["target_days"] == target_days)
            & (timestamp["model"] == model)
        ].replace([np.inf, -np.inf], np.nan).dropna(subset=["observed_atm_iv_percent", "atm_iv_gap_pp"]).copy()
        panel["iv_bin"] = pd.qcut(panel["observed_atm_iv_percent"], q=10, duplicates="drop")
        binned = (
            panel
            .groupby("iv_bin", observed=True)
            .agg(
                observed_atm_iv_percent=("observed_atm_iv_percent", "mean"),
                atm_iv_gap_pp=("atm_iv_gap_pp", "mean"),
                atm_iv_gap_se_pp=("atm_iv_gap_pp", lambda x: float(x.std(ddof=1) / math.sqrt(len(x))) if len(x) > 1 else np.nan),
                n_timestamps=("t_index", "count"),
            )
            .reset_index(drop=True)
        )
        binned["target_days"] = target_days
        binned["model"] = model
        binned_rows.append(binned)

        ax.errorbar(
            binned["observed_atm_iv_percent"],
            binned["atm_iv_gap_pp"],
            yerr=1.96 * binned["atm_iv_gap_se_pp"],
            color=colors[model],
            marker="o",
            linewidth=1.7,
            capsize=2.5,
            label=model,
        )

    ax.axhline(0.0, color="#111827", linewidth=0.8, alpha=0.6)
    ax.set_title(f"{target_days:.0f}-day ATM IV gap", loc="left", fontsize=11)
    ax.set_xlabel("Observed ATM IV (%)")
    ax.grid(True, color="#e5e7eb", linewidth=0.8)

axes[0].set_ylabel("Model minus observed ATM IV (percentage points)")
fig.suptitle("Predictive ATM IV gap by observed ATM IV level", x=0.07, ha="left", fontsize=13)
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", bbox_to_anchor=(0.98, 0.965), ncol=2, frameon=False, fontsize=9)
fig.tight_layout(rect=[0, 0, 1, 0.88])

binned = pd.concat(binned_rows, ignore_index=True)
binned_path = TABLE_DIR / "binned_gap_by_observed_atm_iv.csv"
fig_path = FIG_DIR / "predictive_atm_iv_gap_by_observed_iv_bins.png"
pdf_path = FIG_DIR / "predictive_atm_iv_gap_by_observed_iv_bins.pdf"
binned.to_csv(binned_path, index=False)
fig.savefig(fig_path, bbox_inches="tight", dpi=180)
fig.savefig(pdf_path, bbox_inches="tight")
print(f"Saved {binned_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {fig_path.relative_to(PROJECT_ROOT)}")
print(f"Saved {pdf_path.relative_to(PROJECT_ROOT)}")
plt.show()

## Compact LaTeX-ready table

In [ ]:
latex_rows = []
for _, row in shrinkage.iterrows():
    latex_rows.append({
        "Maturity": f"{row['target_days']:.0f}d",
        "Model": row["model"],
        "Obs. IV coef.": f"{row['observed_iv_coef_alone']:.3f}",
        "Obs. IV p": format_p(row["observed_iv_p_alone"]),
        "Obs. IV coef. + gap": f"{row['observed_iv_coef_with_gap']:.3f}",
        "Gap coef.": f"{row['gap_coef_with_observed_iv']:.3f}",
        "Gap p": format_p(row["gap_p_with_observed_iv"]),
        "Shrinkage": f"{100.0 * row['absolute_observed_iv_coef_shrinkage']:.1f}%",
    })
latex_ready = pd.DataFrame(latex_rows)
latex_ready_path = TABLE_DIR / "deterioration_iv_gap_latex_ready.csv"
latex_ready.to_csv(latex_ready_path, index=False)
print(f"Saved {latex_ready_path.relative_to(PROJECT_ROOT)}")
latex_ready